# ProVe-Arabic — Evaluation

The evaluation harness and every experiment reported in Chapter 5 of the dissertation.
Referenced by Appendix A.6.

**Contents:** the dataset-agnostic scorer, the English passage-level baseline, the controlled
cross-lingual comparison, the claim-level full-pipeline run, the learned Random Forest
aggregator and the relevance ablation, and the Arabic gold-set evaluation.

**Requirements:** Colab with a **T4 GPU**. The model and datasets download automatically on first run. **Run cell 1, restart the runtime when prompted, then continue.**

**This notebook is self-contained**: the setup cells below define every function it uses, so it
can be run on its own and in any order relative to the other notebooks.

**Timings on a T4:** the passage-level baseline scores 2,045 passages (~10 min). The claim-level run scores up to 150 passages for each of 409 references (40 min).
The aggregator experiments load cached feature vectors and complete in seconds.

**WTR is required and is not included.**

See the note before the experiments.

## 1 — Setup  *(restart the runtime after the install cell)*

These cells reproduce the pipeline modules this notebook depends on, so it runs standalone.

In [ ]:
# Pinned environment
!pip install -q --force-reinstall --no-deps "transformers==4.46.3"
!pip install -q pysbd camel-tools beautifulsoup4 lxml requests sentencepiece protobuf sacrebleu

In [ ]:
# Fixed seeds
import os, random, numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
GEN = torch.Generator(); GEN.manual_seed(SEED)      # passed to DataLoader so shuffling is fixed

# For bit-identical GPU results, uncomment the two lines below BEFORE any CUDA call
# They force deterministic kernels, but slow training and raise errors for ops that have no deterministic implementation
# Seeds alone give reproducible convergence
# os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
# torch.use_deterministic_algorithms(True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| seed:', SEED)

# Datasets
# Downloaded once from a public repository and cached locally in PD
from huggingface_hub import hf_hub_download
import shutil, os

DATA_REPO = 'ammarikhan003/prove-arabic-data'
PD = '/content/data'; os.makedirs(PD, exist_ok=True)

FILES = ['arabic_train_pairs_fixed.jsonl',                        # verbaliser training corpus (7,607 pairs)
         'val_fixed.json',                                         # held-out set for chrF
         'arabic_gold_candidates - arabic_gold_candidates.csv',     # annotated Arabic gold set (60 items)
         'wtr_claim_features.json',                                # cached claim-level feature vectors
         'gold_features.json',
         'ar_label_cache.json']                                     # cached gold-set feature vectors

for f in FILES:
    try:
        p = hf_hub_download(repo_id=DATA_REPO, filename=f, repo_type='dataset')
        shutil.copy(p, f'{PD}/{f}')
    except Exception as e:
        print('could not fetch', f, '->', type(e).__name__)
print('data ready in', PD)
for f in sorted(os.listdir(PD)): print('  ', f)

In [ ]:
import os, sys, zipfile, requests
from google.colab import files

PD = '/content/data'
os.makedirs(PD, exist_ok=True)
wtr_path = os.path.join(PD, 'WTR.json')

FIGSHARE_ARTICLE_ID = "21151513"
FIGSHARE_API_URL = f"https://api.figshare.com/v2/articles/{FIGSHARE_ARTICLE_ID}"

def download_from_figshare():
    print(f"Fetching dataset metadata from official Figshare repository (DOI: 10.6084/m9.figshare.{FIGSHARE_ARTICLE_ID})...")
    resp = requests.get(FIGSHARE_API_URL, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    file_list = data.get('files', [])
    print(f"Found {len(file_list)} file(s) on Figshare.")

    for file_info in file_list:
        file_name = file_info.get('name')
        download_url = file_info.get('download_url')
        dest_path = os.path.join(PD, file_name)

        print(f"Downloading {file_name} from Figshare...")
        r = requests.get(download_url, stream=True, timeout=120)
        r.raise_for_status()

        with open(dest_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

        # Unpack if Figshare delivers the dataset compressed (.zip / .tar.gz)
        if file_name.endswith('.zip'):
            print(f"Extracting {file_name}...")
            with zipfile.ZipFile(dest_path, 'r') as zip_ref:
                zip_ref.extractall(PD)
            os.remove(dest_path)

if not os.path.exists(wtr_path):
    try:
        download_from_figshare()
        if os.path.exists(wtr_path):
            print("\n Successfully obtained WTR.json from original Figshare source!")
        else:
            print("\n Downloaded files from Figshare, but WTR.json was not found at expected location.")
    except Exception as e:
        print(f"\n Automatic download from Figshare failed: {e}")
        print("=" * 60)
        print("MANUAL FALLBACK: Please upload `WTR.json` manually.")
        print("Source: https://doi.org/10.6084/m9.figshare.21151513")
        print("=" * 60)
        uploaded = files.upload()
        if 'WTR.json' in uploaded:
            with open(wtr_path, 'wb') as f:
                f.write(uploaded['WTR.json'])
            print(" WTR.json uploaded successfully!")
else:
    print(" WTR.json is already present in /content/data.")

### Stage B module
Fetching, cleaning, language routing, and Arabic-aware segmentation.

In [ ]:
!pip install -q pysbd camel-tools beautifulsoup4 lxml requests
import re, requests, pysbd
from urllib.parse import urlsplit, urlunsplit, quote
from bs4 import BeautifulSoup

# CAMeL normalisation (real role), regex fallback
try:
    from camel_tools.utils.normalize import normalize_alef_ar, normalize_alef_maksura_ar, normalize_teh_marbuta_ar
    from camel_tools.utils.dediac import dediac_ar
    def normalize_ar(t): return normalize_teh_marbuta_ar(normalize_alef_maksura_ar(normalize_alef_ar(dediac_ar(t))))
except Exception:
    def normalize_ar(t):
        t=re.sub(r'[\u064B-\u0652\u0670\u0640]','',t); t=re.sub(r'[إأآا]','ا',t)
        return t.replace('ى','ي').replace('ة','ه')

# language routing
AR=re.compile(r'[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF]')
def arabic_ratio(t):
    L=[c for c in t if c.isalpha()]
    return sum(bool(AR.match(c)) for c in L)/len(L) if L else 0.0
def detect_lang(t): return 'ar' if arabic_ratio(t)>=0.4 else 'en'

# fetch
HEADERS={'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
         '(KHTML, like Gecko) Chrome/124.0 Safari/537.36 ProVe-Arabic-research/1.0',
         'Accept-Language':'ar,en;q=0.9',
         'Accept':'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8'}
def _safe(u):
    p=urlsplit(u); return urlunsplit((p.scheme,p.netloc,quote(p.path),p.query,p.fragment))
def fetch_html(url):
    r=requests.get(_safe(url),headers=HEADERS,timeout=25)
    r.raise_for_status(); r.encoding=r.apparent_encoding or r.encoding
    return r.text

# clean
JUNK_TAGS=['script','style','noscript','nav','footer','header','aside','form','button','svg','iframe','figure','sup']
BLOCKS=['p','li','h1','h2','h3','h4','h5','h6','td','th','blockquote','caption','dd','dt']
LISTTOGGLE=re.compile(r'^\s*القائمة\s*\.{2,}\s*')
PUNCT_FIX=[(re.compile(r'\s+([.,؛،:!؟…\)\]])'),r'\1'),(re.compile(r'([(\[])\s+'),r'\1'),(re.compile(r'\s{2,}'),' ')]
def tidy_spacing(t):
    for p,r in PUNCT_FIX: t=p.sub(r,t)
    return t.strip()
def clean_to_text(html):
    soup=BeautifulSoup(html,'lxml')
    for t in soup(JUNK_TAGS): t.decompose()
    seen,chunks=set(),[]
    for b in soup.find_all(BLOCKS):
        txt=tidy_spacing(LISTTOGGLE.sub('',b.get_text(' ',strip=True)))
        if not txt or txt in seen: continue
        seen.add(txt)
        if txt[-1] not in '.!?؟…؛،:': txt+='.'
        chunks.append(txt)
    text='\n'.join(chunks)
    if len(text)<200: text=tidy_spacing(soup.get_text('\n',strip=True))
    return re.sub(r'\n{2,}','\n',text).strip()

# segment (pysbd + terminal merge)
TERMINALS='.؟?!…؛'
_seg={}
def _segmenter(lang):
    if lang not in _seg: _seg[lang]=pysbd.Segmenter(language=('ar' if lang=='ar' else 'en'),clean=False)
    return _seg[lang]
def segment(text,lang,min_chars=20):
    seg=_segmenter(lang); raw=[]
    for line in text.split('\n'):
        line=line.strip()
        if line: raw+=[s.strip() for s in seg.segment(line) if s.strip()]
    merged,buf=[],''
    for s in raw:
        buf=f'{buf} {s}'.strip() if buf else s
        if buf[-1] in TERMINALS: merged.append(buf); buf=''
    if buf: merged.append(buf)
    seen,out=set(),[]
    for s in merged:
        if s in seen or sum(c.isalpha() for c in s)<5 or len(s)<min_chars: continue
        seen.add(s); out.append(s)
    return out

print("Stage B module ready.")

### Stage A — verbaliser  *(set `RETRAIN` here if you wish to rebuild it)*

In [ ]:
# RETRAIN = False  ->  load the trained model from the Hugging Face Hub (seconds). DEFAULT.
# RETRAIN = True   ->  rebuild it from the training corpus (~15 min on a T4)

# Training writes to a SEPARATE directory, so retraining never overwrites the reference weights
# Both routes define the same verbalise() function

RETRAIN = False

MODEL_REPO = 'ammarikhan003/prove-arabic-mt5'
RETRAIN_DIR = '/content/mt5_retrained'

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

if not RETRAIN:
    TOK = AutoTokenizer.from_pretrained(MODEL_REPO)
    verbaliser = AutoModelForSeq2SeqLM.from_pretrained(MODEL_REPO).to(device).eval()
    print('verbaliser LOADED from', MODEL_REPO)
else:
    import json
    from torch.utils.data import DataLoader
    from transformers import get_linear_schedule_with_warmup

    TOK = AutoTokenizer.from_pretrained('google/mt5-small')
    verbaliser = AutoModelForSeq2SeqLM.from_pretrained('google/mt5-small').to(device)

    pairs = [json.loads(l) for l in open(f'{PD}/arabic_train_pairs_fixed.jsonl')]
    print('training pairs:', len(pairs))

    def collate(batch):
        enc = TOK([x['input'] for x in batch], return_tensors='pt',
                  padding=True, truncation=True, max_length=64)
        lab = TOK(text_target=[x['target'] for x in batch], return_tensors='pt',
                  padding=True, truncation=True, max_length=96)['input_ids']
        lab[lab == TOK.pad_token_id] = -100      # mask padding out of the loss
        enc['labels'] = lab
        return enc

    loader = DataLoader(pairs, batch_size=8, shuffle=True,
                        collate_fn=collate, generator=GEN)   # GEN fixes the shuffle order
    opt = torch.optim.AdamW(verbaliser.parameters(), lr=3e-4)
    steps = len(loader) * 3
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)

    verbaliser.train()
    for ep in range(3):
        tot = 0.0
        for i, b in enumerate(loader):
            b = {k: v.to(device) for k, v in b.items()}
            loss = verbaliser(**b).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(verbaliser.parameters(), 1.0)   # prevents loss spikes
            opt.step(); sched.step(); opt.zero_grad()
            tot += loss.item()
            if i % 200 == 0: print(f'  ep{ep+1} step {i}/{len(loader)} loss {loss.item():.3f}')
        print(f'epoch {ep+1}/3 avg loss {tot/len(loader):.3f}')
    verbaliser.eval()
    verbaliser.save_pretrained(RETRAIN_DIR); TOK.save_pretrained(RETRAIN_DIR)
    print('verbaliser RETRAINED and saved to', RETRAIN_DIR)

def verbalise(subj, prop, obj):
    """Triple -> Arabic sentence. Beam search for fluency; the two repetition
    constraints suppress a failure mode where a phrase repeated within one sentence."""
    enc = TOK(f'{subj} | {prop} | {obj}', return_tensors='pt',
              truncation=True, max_length=64).to(device)
    with torch.no_grad():
        o = verbaliser.generate(**enc, max_length=96, num_beams=4,
                                no_repeat_ngram_size=3, repetition_penalty=1.2)
    return TOK.decode(o[0], skip_special_tokens=True)

print('smoke test:', verbalise('دوغلاس آدمز', 'مكان الولادة', 'كامبريدج'))

### Stage C/D — entailment
Passage as premise, claim as hypothesis. Class mapping read from the model config.

In [ ]:
import torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = 'cuda' if torch.cuda.is_available() else 'cpu'
NLI = 'MoritzLaurer/mDeBERTa-v3-base-mnli-xnli'
nli_tok = AutoTokenizer.from_pretrained(NLI)
nli = AutoModelForSequenceClassification.from_pretrained(NLI).to(device).eval()
id2label = {int(k): v.lower() for k, v in nli.config.id2label.items()}   # read labels from the model, don't assume order
print('device:', device, '| label map:', id2label)

def entail_scores(premise, hypothesis):
    """premise = evidence passage, hypothesis = the claim. Returns {entailment, neutral, contradiction} probs."""
    x = nli_tok(premise, hypothesis, return_tensors='pt', truncation=True, max_length=256).to(device)
    with torch.no_grad():
        probs = F.softmax(nli(**x).logits[0], dim=-1)
    return {id2label[i]: probs[i].item() for i in range(len(probs))}

def verify(claim, passages, topk=5, thresh=0.5):
    scored = []
    for p in passages:
        s = entail_scores(p, claim)
        e, c, n = s.get('entailment',0), s.get('contradiction',0), s.get('neutral',0)
        scored.append({'entail': e, 'contradict': c, 'neutral': n, 'passage': p})
    scored.sort(key=lambda r: max(r['entail'], r['contradict']), reverse=True)   # most-engaged first
    top = scored[:topk]
    best_e = max((r['entail'] for r in top), default=0)
    best_c = max((r['contradict'] for r in top), default=0)
    if max(best_e, best_c) < thresh: verdict = 'NOT ENOUGH INFO'
    elif best_e >= best_c:           verdict = 'SUPPORTS'
    else:                             verdict = 'REFUTES'
    return verdict, top

## 2 — The scorer

Dataset-agnostic: it sees only gold and predicted labels, so the same code scores the English
benchmark, the silver Arabic set, and the gold set. Differences between reported results
therefore cannot be artefacts of differing metric implementations.

`unprocessable_as` exposes the evaluation policy for references that yield no prediction
(Section 5.1.3).

In [ ]:
# Only needs records with 'gold' + 'pred'
# Same scorer for the sample now and WTR later
import numpy as np, pandas as pd
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

LABELS = ['SUPPORTS', 'REFUTES', 'NEI']

def evaluate(records, unprocessable_as='NEI', title='evaluation'):
    """records: list of dicts with 'gold', 'pred' (None if reference unprocessable),
       and optionally 'support_prob'."""
    n = len(records)
    proc = [r for r in records if r['pred'] is not None]
    coverage = len(proc) / n if n else 0
    gold = [r['gold'] for r in records]
    pred = [r['pred'] if r['pred'] is not None else unprocessable_as for r in records]

    print(f"===== {title} =====")
    print(f"items: {n}   |   references processable: {len(proc)}/{n} ({coverage:.0%})")
    print(f"(unprocessable references scored operationally as '{unprocessable_as}')\n")
    print("accuracy:", round(accuracy_score(gold, pred), 3), "\n")
    print(classification_report(gold, pred, labels=LABELS, zero_division=0))

    cm = pd.DataFrame(confusion_matrix(gold, pred, labels=LABELS),
                      index=[f'gold:{l}' for l in LABELS],
                      columns=[f'pred:{l}' for l in LABELS])
    print("confusion matrix:\n", cm, "\n")

    # support-probability spread per gold class, the input to the Random Forest decision later
    if any(r.get('support_prob') is not None for r in proc):
        print("support_prob by gold class (processable only):")
        for l in LABELS:
            ys = [r['support_prob'] for r in proc if r['gold'] == l and r.get('support_prob') is not None]
            if ys:
                print(f"  {l:9s} n={len(ys):3d}  mean={np.mean(ys):.2f}  min={min(ys):.2f}  max={max(ys):.2f}")
    return cm

# self-test on synthetic data so every metric is visibly exercised
_demo = [
    {'gold':'SUPPORTS','pred':'SUPPORTS','support_prob':0.95},
    {'gold':'SUPPORTS','pred':'SUPPORTS','support_prob':0.88},
    {'gold':'SUPPORTS','pred':'REFUTES','support_prob':0.20},   # a miss
    {'gold':'REFUTES', 'pred':'REFUTES','support_prob':0.10},
    {'gold':'REFUTES', 'pred':'NEI','support_prob':0.40},       # a miss
    {'gold':'NEI',     'pred':'NEI','support_prob':0.50},
    {'gold':'NEI',     'pred':None,'support_prob':None},        # unprocessable reference
]
evaluate(_demo, title='self-test (synthetic)')

## WTR benchmark — must be obtained separately

The experiments in this notebook evaluate against **WTR** (Wikidata Textual References), the
benchmark introduced with ProVe: 409 Wikidata triples paired with their real references,
annotated at passage level, shipped with the archived HTML of each reference.

**WTR is not redistributed here.** It is the work of the ProVe authors and is thus taken from the original source earlier.

The next cell checks
for it and reports clearly if it is absent, rather than failing later with an obscure error.

In [ ]:
WTR_PATH = f'{PD}/WTR.json'
if os.path.exists(WTR_PATH):
    import json
    _w = json.load(open(WTR_PATH))
    if isinstance(_w, dict): _w = list(_w.values())
    print(f'WTR found: {len(_w)} triple-reference pairs')
else:
    print('WTR.json NOT FOUND in', PD)
    print('The experiments in this notebook require it. Obtain it from the ProVe authors')
    print('(see notebook header) and upload it to', PD)

## 3 — WTR structure

Establishes the benchmark's record layout, its label encoding, and the availability of archived
HTML, before anything depends on them.

In [ ]:
import json, pprint

PATH = f'{PD}/WTR.json'

# load, handling array / dict / json-lines
try:
    data = json.load(open(PATH))
    if isinstance(data, dict):
        data = data.get('data', list(data.values())) if 'data' in data else list(data.values())
except json.JSONDecodeError:
    data = [json.loads(l) for l in open(PATH) if l.strip()]

print("records:", len(data))
r0 = data[0]
print("record type:", type(r0).__name__)
print("top-level keys:", list(r0.keys()) if isinstance(r0, dict) else "(not a dict)")
print("\n===== one full record =====")
pprint.pprint(r0, depth=4, width=120)

In [ ]:
from collections import Counter
def maj_labels(rec):
    out=[]
    for blk in rec.get('crowd_annotations_T1',[]):
        if isinstance(blk,dict) and 'relation_maj' in blk: out.append(blk['relation_maj'])
    t2=rec.get('crowd_annotations_T2')
    if isinstance(t2,dict) and 'relation_maj' in t2: out.append(t2['relation_maj'])
    return out
all_labels=[l for r in data for l in maj_labels(r)]
print("label counts:", Counter(all_labels))
# show one evidence sentence per label value
seen={}
for r in data:
    for blk in r.get('crowd_annotations_T1',[]):
        m=blk.get('relation_maj')
        if m not in seen and blk.get('evidence'):
            seen[m]=blk['evidence'][:120]
print()
for k in sorted(seen): print(f"label {k}: {seen[k]}")

In [ ]:
import glob, os, json
from collections import Counter

PD = PD

# 1. find the html folder and see how files are named
dirs = [p for p in glob.glob(f'{PD}/*') if os.path.isdir(p)]
print("folders in prove_arabic:", [os.path.basename(d) for d in dirs])

# look for the folder containing .html files
html_dir = None
for d in dirs + [PD]:
    h = glob.glob(f'{d}/*.html')
    if h:
        html_dir = d
        print(f"\nHTML folder: {d}  ({len(h)} html files)")
        print("sample filenames:", [os.path.basename(x) for x in h[:3]])
        break

if not html_dir:
    print("\n⚠ no .html files found — tell me the folder name")

# 2. do the record 'html' fields match actual files?
data = json.load(open(f'{PD}/WTR.json'))
if isinstance(data, dict):
    data = list(data.values())

if html_dir:
    have = set(os.path.basename(x) for x in glob.glob(f'{html_dir}/*.html'))
    want = [r['html'] for r in data if r.get('html')]
    match = sum(1 for w in want if w in have)
    print(f"\nrecords whose html file is present: {match}/{len(want)}")

# 3. confirm the label encoding (0/1/2) across the whole set
def maj_labels(rec):
    out = []
    for blk in rec.get('crowd_annotations_T1', []):
        if isinstance(blk, dict) and 'relation_maj' in blk:
            out.append(blk['relation_maj'])
    t2 = rec.get('crowd_annotations_T2')
    if isinstance(t2, dict) and 'relation_maj' in t2:
        out.append(t2['relation_maj'])
    return out

print("\nlabel distribution:", Counter(l for r in data for l in maj_labels(r)))

## 4 — Experiment 1: English passage-level baseline

Measures the entailment stage alone against human passage annotations, bypassing Stages A and B.
Uses only the per-passage (T1) annotations; the collective annotation cannot be attributed to any
individual passage. Expect **2,045 passages** and accuracy **0.5510** (argmax) / **0.6020**
(thresholded).

In [ ]:
# Needs in memory: entail_scores (Stage C/D cell) + evaluate (scorer cell). No Stage A/B/E.
import json
from collections import Counter

PD = PD
data = json.load(open(f'{PD}/WTR.json'))
if isinstance(data, dict):
    data = list(data.values())

# WTR label encoding -> pipeline labels
WTR2LAB = {0: 'SUPPORTS', 1: 'REFUTES', 2: 'NEI'}
# your entailment classes -> same three
ENT2LAB = {'entailment': 'SUPPORTS', 'contradiction': 'REFUTES', 'neutral': 'NEI'}

def gold_passages(rec):
    """yield (evidence_text, gold_label) per annotated passage, using majority vote."""
    out = []
    for blk in rec.get('crowd_annotations_T1', []):
        if isinstance(blk, dict) and blk.get('evidence') and 'relation_maj' in blk:
            out.append((blk['evidence'], blk['relation_maj']))
    t2 = rec.get('crowd_annotations_T2')
    if isinstance(t2, dict) and t2.get('evidence') and 'relation_maj' in t2:
        ev = t2['evidence']
        if isinstance(ev, list):  # T2 stores a list of evidence
            for e in ev:
                out.append((e, t2['relation_maj']))
        else:
            out.append((ev, t2['relation_maj']))
    return out

def pred_label(s, refute_thresh=0.7):
    e, c, n = s.get('entailment', 0), s.get('contradiction', 0), s.get('neutral', 0)
    if e >= c and e >= n:  # entailment wins -> SUPPORTS
        return 'SUPPORTS'
    if c >= refute_thresh:  # only a CONFIDENT contradiction counts as REFUTES
        return 'REFUTES'
    return 'NEI'  # everything else (incl. weak/ambiguous contradiction) -> NEI

records = []
for rec in data:
    claim = f"The {rec.get('property_label', '')} of {rec.get('entity_label', '')} is {rec.get('object_label', '')}."
    if not claim:
        continue
    for evidence, wtr_lab in gold_passages(rec):
        if wtr_lab not in WTR2LAB or not evidence.strip():
            continue
        s = entail_scores(evidence, claim)  # premise=evidence, hypothesis=claim
        records.append({
            'gold': WTR2LAB[wtr_lab],
            'pred': pred_label(s),
            'support_prob': s.get('entailment', 0.0)
        })

print("passages scored:", len(records))
print("gold distribution:", Counter(r['gold'] for r in records))
evaluate(records, title='WTR passage-level (Stage C/D, English)')

### Thresholded variant

Predicts REFUTES only on a confident contradiction (≥ 0.70), else NEI. An ablation at a single
pre-chosen threshold, not a tuned result.

In [ ]:
import json, statistics as st
from collections import Counter

data = json.load(open(f'{PD}/WTR.json'))
if isinstance(data, dict):
    data = list(data.values())

WTR2LAB = {0: 'SUPPORTS', 1: 'REFUTES', 2: 'NEI'}

def pred_label(s, refute_thresh=0.7):
    e, c, n = s.get('entailment', 0), s.get('contradiction', 0), s.get('neutral', 0)
    if e >= c and e >= n:
        return 'SUPPORTS'
    if c >= refute_thresh:
        return 'REFUTES'
    return 'NEI'

recs = []
for rec in data:
    claim = f"The {rec.get('property_label', '')} of {rec.get('entity_label', '')} is {rec.get('object_label', '')}."
    for blk in rec.get('crowd_annotations_T1', []):  # T1 ONLY
        if not (isinstance(blk, dict) and blk.get('evidence') and blk.get('relation_maj') in WTR2LAB):
            continue
        s = entail_scores(blk['evidence'], claim)
        recs.append({
            'gold': WTR2LAB[blk['relation_maj']],
            'pred': pred_label(s),
            'support_prob': s.get('entailment', 0.0)
        })

print("T1-only passages:", len(recs), Counter(r['gold'] for r in recs))
evaluate(recs, title='WTR T1-only, thresholded (refute>=0.7)')

### Per-example diagnostic

Inspects individual scored passages. Distinguishes wiring error from a
model failure.

In [ ]:
import json
from collections import Counter

data = json.load(open(f'{PD}/WTR.json'))
if isinstance(data, dict):
    data = list(data.values())

WTR2LAB = {0: 'SUPPORTS', 1: 'REFUTES', 2: 'NEI'}
ENT2LAB = {'entailment': 'SUPPORTS', 'contradiction': 'REFUTES', 'neutral': 'NEI'}

recs = []
for rec in data:
    claim = f"The {rec.get('property_label', '')} of {rec.get('entity_label', '')} is {rec.get('object_label', '')}."
    for blk in rec.get('crowd_annotations_T1', []):  # T1 ONLY, individual passages
        if not (isinstance(blk, dict) and blk.get('evidence') and blk.get('relation_maj') in WTR2LAB):
            continue
        s = entail_scores(blk['evidence'], claim)
        recs.append({
            'gold': WTR2LAB[blk['relation_maj']],
            'pred': ENT2LAB[max(s, key=s.get)],
            'e': s.get('entailment', 0),
            'c': s.get('contradiction', 0),
            'n': s.get('neutral', 0)
        })

print("T1-only passages:", len(recs), Counter(r['gold'] for r in recs))

for lab in ['SUPPORTS', 'REFUTES', 'NEI']:
    sub = [r for r in recs if r['gold'] == lab]
    if sub:
        import statistics as st
        print(f"{lab:9s} n={len(sub):4d} | mean entail={st.mean(r['e'] for r in sub):.2f} "
              f"contra={st.mean(r['c'] for r in sub):.2f} neutral={st.mean(r['n'] for r in sub):.2f}")

evaluate(recs, title='WTR T1-only passage-level')

## 5 — Experiment 2: the controlled cross-lingual comparison

**The central experiment.** Identical passages, gold labels, model, and decision rule. Only the
language of the claim varies. Expect a fully-Arabic subset of **355 passages** and supporting-class
F1 of **0.6400** (English) against **0.6200** (Arabic).

In [ ]:
import requests, json, time
from collections import Counter

PD = PD
UA = {'User-Agent': 'ProVe-Arabic-thesis/1.0'}
data = json.load(open(f'{PD}/WTR.json'))
if isinstance(data, dict):
    data = list(data.values())

WTR2LAB = {0: 'SUPPORTS', 1: 'REFUTES', 2: 'NEI'}

# 1. gather all Q/P ids needing Arabic labels
ids = set()
for rec in data:
    for k in ('entity_id', 'property_id'):
        if rec.get(k):
            ids.add(rec[k])
    dv = rec.get('datavalue') or {}
    if isinstance(dv, dict) and dv.get('type') == 'wikibase-entityid':
        oid = dv.get('value', {}).get('id')
        if oid:
            ids.add(oid)

ids = [x for x in ids if x]
print('unique ids to resolve:', len(ids))

# 2. fetch Arabic labels from Wikidata
LAB_CACHE = f'{PD}/ar_label_cache.json'

# start from whatever is cached (values may be None = confirmed no Arabic label)
cache = json.load(open(LAB_CACHE)) if os.path.exists(LAB_CACHE) else {}
missing = [i for i in ids if i not in cache]
print(f'cache has {len(cache)} entries | {len(missing)} of {len(ids)} WTR ids need resolving')

if missing:
    failed = []
    for i in range(0, len(missing), 50):
        chunk = missing[i:i+50]
        ents = fetch_batch(chunk)
        if not ents:
            failed.extend(chunk); continue
        for qid in chunk:
            ent = ents.get(qid, {})
            cache[qid] = ent.get('labels', {}).get('ar', {}).get('value')
        time.sleep(1.0)

    json.dump(cache, open(LAB_CACHE, 'w'))          # SAVE FIRST, always

    # retry the failures individually — batch failures are usually one bad id
    if failed:
        print(f'{len(failed)} ids failed in batch; retrying individually...')
        still = []
        for qid in failed:
            ents = fetch_batch([qid])
            if ents:
                cache[qid] = ents.get(qid, {}).get('labels', {}).get('ar', {}).get('value')
            else:
                still.append(qid)
            time.sleep(0.5)
        json.dump(cache, open(LAB_CACHE, 'w'))
        if still:
            print(f'WARNING: {len(still)} ids still unresolved: {still[:10]}')
            print('These are treated as having no Arabic label, which may slightly '
                  'understate the fully-Arabic subset. Re-run this cell to retry them.')

# only non-None values count as a resolved Arabic label
ar_label = {k: v for k, v in cache.items() if v}
resolved = sum(1 for i in ids if ar_label.get(i))
print(f'Arabic labels resolved: {resolved}/{len(ids)} ({resolved/len(ids):.0%})')


# 3. build Arabic + English claim per record
def build(rec):
    s_qid, p_id = rec.get('entity_id'), rec.get('property_id')
    dv = rec.get('datavalue') or {}
    o_ent = isinstance(dv, dict) and dv.get('type') == 'wikibase-entityid'
    o_qid = dv.get('value', {}).get('id') if o_ent else None
    s_ar = ar_label.get(s_qid) or rec.get('entity_label', '')
    p_ar = ar_label.get(p_id) or rec.get('property_label', '')
    o_ar = (ar_label.get(o_qid) if o_ent else rec.get('object_label', '')) or rec.get('object_label', '')
    fully = bool(ar_label.get(s_qid)) and bool(ar_label.get(p_id)) and (bool(ar_label.get(o_qid)) if o_ent else True)
    claim_ar = verbalise(s_ar, p_ar, o_ar)
    claim_en = f"The {rec.get('property_label', '')} of {rec.get('entity_label', '')} is {rec.get('object_label', '')}."
    return claim_ar, claim_en, fully

def pl(e, c, n, t=0.7):
    if e >= c and e >= n:
        return 'SUPPORTS'
    if c >= t:
        return 'REFUTES'
    return 'NEI'

recs = []
for rec in data:
    claim_ar, claim_en, fully = build(rec)
    for blk in rec.get('crowd_annotations_T1', []):
        if not (isinstance(blk, dict) and blk.get('evidence') and blk.get('relation_maj') in WTR2LAB):
            continue
        ev = blk['evidence']
        a = entail_scores(ev, claim_ar)
        en = entail_scores(ev, claim_en)
        recs.append({
            'gold': WTR2LAB[blk['relation_maj']],
            'fully': fully,
            'ar': (a.get('entailment', 0), a.get('contradiction', 0), a.get('neutral', 0)),
            'en': (en.get('entailment', 0), en.get('contradiction', 0), en.get('neutral', 0))
        })

print('\npassages:', len(recs), '| fully-Arabic-claim passages:', sum(r['fully'] for r in recs))

def report(subset, lang, title):
    argmax = [{
        'gold': r['gold'],
        'pred': max((('SUPPORTS', r[lang][0]), ('REFUTES', r[lang][1]), ('NEI', r[lang][2])), key=lambda x: x[1])[0],
        'support_prob': r[lang][0]
    } for r in subset]
    thr = [{
        'gold': r['gold'],
        'pred': pl(*r[lang]),
        'support_prob': r[lang][0]
    } for r in subset]
    evaluate(argmax, title=f'{title} — argmax')
    evaluate(thr, title=f'{title} — thresholded')

print('\n########## ENGLISH claim vs EN passage [ALL] — should reproduce your baseline ##########')
report(recs, 'en', 'EN [ALL]')

print('\n########## ARABIC claim vs EN passage [ALL] ##########')
report(recs, 'ar', 'AR [ALL]')

sub = [r for r in recs if r['fully']]
print(f'\n########## CONTROLLED head-to-head on fully-Arabic subset (n={len(sub)}) ##########')
report(sub, 'en', 'EN [fully-AR subset]')
report(sub, 'ar', 'AR [fully-AR subset]')

### Arabic label resolution

Batched Wikidata API lookups with backoff. Pacing matters: without it, batches fail silently and
the Arabic subset is quietly under-counted.

In [ ]:
import time, requests
API='https://www.wikidata.org/w/api.php'; UA={'User-Agent':'ProVe-Arabic-thesis/1.0'}
def fetch_batch(chunk, tries=5):
    for t in range(tries):
        try:
            r=requests.get(API,params={'action':'wbgetentities','ids':'|'.join(chunk),
                           'props':'labels','languages':'ar','format':'json'},headers=UA,timeout=90)
            if r.status_code==200 and r.text.strip().startswith('{'):
                return r.json().get('entities',{})
        except Exception:
            pass
        time.sleep(2*(t+1))            # 2,4,6,8,10s — ride out the throttle window
    print("  batch failed after retries:", chunk[0], "…"); return {}

ar_label={}
for i in range(0,len(ids),50):
    for qid,ent in fetch_batch(ids[i:i+50]).items():
        v=ent.get('labels',{}).get('ar',{}).get('value')
        if v: ar_label[qid]=v
    time.sleep(1.0)                    # gentler pacing between batches
print(f'resolved Arabic labels: {len(ar_label)}/{len(ids)} ({len(ar_label)/len(ids):.0%})')

### Rebuild and report

In [ ]:
recs=[]
for rec in data:
    claim_ar,claim_en,fully=build(rec)
    for blk in rec.get('crowd_annotations_T1',[]):
        if not (isinstance(blk,dict) and blk.get('evidence') and blk.get('relation_maj') in WTR2LAB): continue
        ev=blk['evidence']
        a=entail_scores(ev,claim_ar); en=entail_scores(ev,claim_en)
        recs.append({'gold':WTR2LAB[blk['relation_maj']],'fully':fully,
                     'ar':(a.get('entailment',0),a.get('contradiction',0),a.get('neutral',0)),
                     'en':(en.get('entailment',0),en.get('contradiction',0),en.get('neutral',0))})
print('passages:',len(recs),'| fully-Arabic-claim passages:',sum(r['fully'] for r in recs))

sub=[r for r in recs if r['fully']]
print(f'\n########## CONTROLLED head-to-head, fully-Arabic subset (n={len(sub)}) ##########')
report(sub,'en','EN [fully-AR subset]')
report(sub,'ar','AR [fully-AR subset]')

## 6 — Experiment 3: claim-level full pipeline

Stage B retrieval in the loop, reading WTR's archived HTML, with verdicts aggregated per
reference.

Expect **28 of 409** references (6.80%) to yield fewer than three passages. This is roughly 40 minutes on a T4.

In [ ]:
import os, json
from collections import Counter

# Uses local PD path from setup cells
HTML = f'{PD}/htmls' if os.path.exists(f'{PD}/htmls') else PD

data = json.load(open(f'{PD}/WTR.json'))
if isinstance(data, dict):
    data = list(data.values())

WTR2LAB = {0: 'SUPPORTS', 1: 'REFUTES', 2: 'NEI'}

def derived_gold(rec):
    """claim-level gold: any SUPPORTS -> SUPPORTS; else any REFUTES -> REFUTES; else NEI."""
    labs = [
        WTR2LAB[b['relation_maj']]
        for b in rec.get('crowd_annotations_T1', [])
        if isinstance(b, dict) and b.get('relation_maj') in WTR2LAB
    ]
    if not labs:
        return None
    if 'SUPPORTS' in labs:
        return 'SUPPORTS'
    if 'REFUTES' in labs:
        return 'REFUTES'
    return 'NEI'

def aggregate_rule(scored, sup_t=0.5, ref_t=0.9):
    best_e = max((s[0] for s in scored), default=0.0)
    best_c = max((s[1] for s in scored), default=0.0)
    if best_e >= sup_t:
        return 'SUPPORTS', best_e
    if best_c >= ref_t and best_c > best_e:  # confident and stronger than any support
        return 'REFUTES', 0.0
    return 'NEI', best_e

recs = []
fail = Counter()

for i, rec in enumerate(data):
    gold = derived_gold(rec)
    if gold is None:
        fail['no_gold'] += 1
        continue

    claim = f"The {rec.get('property_label', '')} of {rec.get('entity_label', '')} is {rec.get('object_label', '')}."
    path = os.path.join(HTML, rec.get('html', ''))
    r = {'gold': gold, 'pred': None, 'support_prob': None, 'n_passages': 0}

    try:
        if not os.path.exists(path):
            fail['missing_html'] += 1
            recs.append(r)
            continue

        html = open(path, 'rb').read()  # SAVED html, no dead links
        text = clean_to_text(html)
        passages = segment(text, detect_lang(text))
        r['n_passages'] = len(passages)

        if len(passages) >= 3:
            scored = []
            for p in passages[:150]:  # cap for runtime
                s = entail_scores(p, claim)
                scored.append((s.get('entailment', 0), s.get('contradiction', 0), s.get('neutral', 0)))
            r['pred'], r['support_prob'] = aggregate_rule(scored)
        else:
            fail['too_few_passages'] += 1
    except Exception as e:
        fail[type(e).__name__] += 1

    recs.append(r)
    if (i + 1) % 50 == 0:
        print(f'  {i + 1}/{len(data)} processed')

print('\nreferences:', len(recs), '| retrieval failures:', dict(fail))
print('gold distribution:', Counter(r['gold'] for r in recs))
evaluate(recs, title='WTR claim-level (FULL PIPELINE: Stage B -> C/D -> E)')

## 7 — Experiment 4: the learned aggregator

Loads the cached per-reference feature vectors so the aggregator can be trained and compared
without repeating the entailment pass.

Expect rule-based **0.4820** against Random Forest
**0.6800** accuracy under five-fold stratified cross-validation.

In [ ]:
import os, json, numpy as np
from collections import Counter

# Uses local PD path from setup cells
HTML = f'{PD}/htmls' if os.path.exists(f'{PD}/htmls') else PD

data = json.load(open(f'{PD}/WTR.json'))
if isinstance(data, dict):
    data = list(data.values())

WTR2LAB = {0: 'SUPPORTS', 1: 'REFUTES', 2: 'NEI'}

def derived_gold(rec):
    """claim-level gold: any SUPPORTS -> SUPPORTS; else any REFUTES -> REFUTES; else NEI."""
    labs = [
        WTR2LAB[b['relation_maj']]
        for b in rec.get('crowd_annotations_T1', [])
        if isinstance(b, dict) and b.get('relation_maj') in WTR2LAB
    ]
    if not labs:
        return None
    if 'SUPPORTS' in labs:
        return 'SUPPORTS'
    if 'REFUTES' in labs:
        return 'REFUTES'
    return 'NEI'

def featurise(scored):
    """summarise a page's per-passage (e,c,n) scores into fixed-length features"""
    if not scored:
        return None
    e = np.array([s[0] for s in scored])
    c = np.array([s[1] for s in scored])
    n = np.array([s[2] for s in scored])

    es = np.sort(e)[::-1]
    cs = np.sort(c)[::-1]
    top = lambda a, k: float(a[k]) if len(a) > k else 0.0

    return [
        float(e.max()), float(e.mean()), top(es, 1), top(es, 2),        # support evidence
        float(c.max()), float(c.mean()), top(cs, 1), top(cs, 2),        # refute evidence
        float(n.mean()),                                                # how neutral overall
        float((e > 0.5).sum()), float((c > 0.5).sum()),                 # counts of strong passages
        float(len(scored)),                                             # page length
        float(e.max() - c.max())                                        # support vs refute margin
    ]

rows = []
for i, rec in enumerate(data):
    gold = derived_gold(rec)
    if gold is None:
        continue

    claim = f"The {rec.get('property_label', '')} of {rec.get('entity_label', '')} is {rec.get('object_label', '')}."
    path = os.path.join(HTML, rec.get('html', ''))
    scored = []

    try:
        if os.path.exists(path):
            html = open(path, 'rb').read()
            text = clean_to_text(html)
            passages = segment(text, detect_lang(text))
            for p in passages[:150]:
                s = entail_scores(p, claim)
                scored.append((s.get('entailment', 0), s.get('contradiction', 0), s.get('neutral', 0)))
    except Exception:
        scored = []

    rows.append({'gold': gold, 'feats': featurise(scored), 'n_passages': len(scored)})
    if (i + 1) % 50 == 0:
        print(f'  {i + 1}/{len(data)}')

output_path = f'{PD}/wtr_claim_features.json'
json.dump(rows, open(output_path, 'w'))

print('\ncached ->', output_path)
print('usable (>=1 passage):', sum(1 for r in rows if r['feats']), '/', len(rows))
print('gold:', Counter(r['gold'] for r in rows))

### Rule versus learned aggregator

In [ ]:
import json, numpy as np
from collections import Counter
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict

PD = PD
rows = json.load(open(f'{PD}/wtr_claim_features.json'))
usable = [r for r in rows if r['feats']]
X = np.array([r['feats'] for r in usable])
y = np.array([r['gold'] for r in usable])
print('training on', len(usable), 'references |', Counter(y))

# the RULE, recomputed from the same features, for a fair head-to-head
def rule_pred(f, sup_t=0.5, ref_t=0.9):
    best_e, best_c = f[0], f[4]
    if best_e >= sup_t:
        return 'SUPPORTS'
    if best_c >= ref_t and best_c > best_e:
        return 'REFUTES'
    return 'NEI'

rule = [{'gold': g, 'pred': rule_pred(f), 'support_prob': f[0]} for f, g in zip(X, y)]

# the RANDOM FOREST, cross-validated (never predicts on data it trained on)
rf = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                            min_samples_leaf=2, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pred = cross_val_predict(rf, X, y, cv=cv)
prob = cross_val_predict(rf, X, y, cv=cv, method='predict_proba')
classes = sorted(set(y))
si = classes.index('SUPPORTS')
forest = [{'gold': g, 'pred': p, 'support_prob': float(pr[si])} for g, p, pr in zip(y, pred, prob)]

evaluate(rule, title='STAGE E — rule-based (hand-written thresholds)')
evaluate(forest, title='STAGE E — Random Forest (5-fold cross-validated)')

# which evidence signals actually matter?
rf.fit(X, y)
names = [
    'max_entail', 'mean_entail', '2nd_entail', '3rd_entail',
    'max_contra', 'mean_contra', '2nd_contra', '3rd_contra',
    'mean_neutral', 'n_strong_entail', 'n_strong_contra',
    'n_passages', 'entail_minus_contra'
]

print('\nfeature importances:')
for n, imp in sorted(zip(names, rf.feature_importances_), key=lambda x: -x[1]):
    print(f'  {n:20s} {imp:.3f}')

In [ ]:
import joblib
joblib.dump(rf, f'{PD}/stage_e_rf.joblib')
print('saved | classes:', rf.classes_)

### Bilingual features and the Arabic claim-level attempt

Underpowered at n=25

In [ ]:
import os, json, time, requests, numpy as np
from collections import Counter

HTML = f'{PD}/htmls' if os.path.exists(f'{PD}/htmls') else PD

data = json.load(open(f'{PD}/WTR.json'))
if isinstance(data, dict):
    data = list(data.values())

WTR2LAB = {0: 'SUPPORTS', 1: 'REFUTES', 2: 'NEI'}
API = 'https://www.wikidata.org/w/api.php'
UA = {'User-Agent': 'ProVe-Arabic-thesis/1.0'}

# 1. Resolve Arabic labels from Wikidata
ids = set()
for rec in data:
    for k in ('entity_id', 'property_id'):
        if rec.get(k):
            ids.add(rec[k])
    dv = rec.get('datavalue') or {}
    if isinstance(dv, dict) and dv.get('type') == 'wikibase-entityid':
        oid = dv.get('value', {}).get('id')
        if oid:
            ids.add(oid)

ids = [x for x in ids if x]

def fetch_batch(chunk, tries=5):
    for t in range(tries):
        try:
            r = requests.get(
                API,
                params={
                    'action': 'wbgetentities',
                    'ids': '|'.join(chunk),
                    'props': 'labels',
                    'languages': 'ar',
                    'format': 'json'
                },
                headers=UA,
                timeout=90
            )
            if r.status_code == 200 and r.text.strip().startswith('{'):
                return r.json().get('entities', {})
        except Exception:
            pass
        time.sleep(2 * (t + 1))
    return {}

LAB_CACHE = f'{PD}/ar_label_cache.json'

# start from whatever is cached (values may be None = confirmed no Arabic label)
cache = json.load(open(LAB_CACHE)) if os.path.exists(LAB_CACHE) else {}
missing = [i for i in ids if i not in cache]
print(f'cache has {len(cache)} entries | {len(missing)} of {len(ids)} WTR ids need resolving')

if missing:
    failed = []
    for i in range(0, len(missing), 50):
        chunk = missing[i:i+50]
        ents = fetch_batch(chunk)
        if not ents:
            failed.extend(chunk); continue
        for qid in chunk:
            ent = ents.get(qid, {})
            cache[qid] = ent.get('labels', {}).get('ar', {}).get('value')
        time.sleep(1.0)

    json.dump(cache, open(LAB_CACHE, 'w'))          # Save first

    # retry the failures individually
    # batch failures are usually one bad id
    if failed:
        print(f'{len(failed)} ids failed in batch; retrying individually...')
        still = []
        for qid in failed:
            ents = fetch_batch([qid])
            if ents:
                cache[qid] = ents.get(qid, {}).get('labels', {}).get('ar', {}).get('value')
            else:
                still.append(qid)
            time.sleep(0.5)
        json.dump(cache, open(LAB_CACHE, 'w'))
        if still:
            print(f'WARNING: {len(still)} ids still unresolved: {still[:10]}')
            print('These are treated as having no Arabic label, which may slightly '
                  'understate the fully-Arabic subset. Re-run this cell to retry them.')

# only non-None values count as an Arabic label
ar_label = {k: v for k, v in cache.items() if v}
resolved = sum(1 for i in ids if ar_label.get(i))
print(f'Arabic labels resolved: {resolved}/{len(ids)} ({resolved/len(ids):.0%})')

# 2. Helpers
def derived_gold(rec):
    """claim-level gold: any SUPPORTS -> SUPPORTS; else any REFUTES -> REFUTES; else NEI."""
    labs = [
        WTR2LAB[b['relation_maj']]
        for b in rec.get('crowd_annotations_T1', [])
        if isinstance(b, dict) and b.get('relation_maj') in WTR2LAB
    ]
    if not labs:
        return None
    if 'SUPPORTS' in labs:
        return 'SUPPORTS'
    if 'REFUTES' in labs:
        return 'REFUTES'
    return 'NEI'

def claims(rec):
    s_qid, p_id = rec.get('entity_id'), rec.get('property_id')
    dv = rec.get('datavalue') or {}
    o_ent = isinstance(dv, dict) and dv.get('type') == 'wikibase-entityid'
    o_qid = dv.get('value', {}).get('id') if o_ent else None

    s_ar = ar_label.get(s_qid) or rec.get('entity_label', '')
    p_ar = ar_label.get(p_id) or rec.get('property_label', '')
    o_ar = (ar_label.get(o_qid) if o_ent else rec.get('object_label', '')) or rec.get('object_label', '')

    fully = (bool(ar_label.get(s_qid))
         and bool(ar_label.get(p_id))
         and bool(ar_label.get(o_qid) if o_ent else None))

    ar = verbalise(s_ar, p_ar, o_ar)
    en = f"The {rec.get('property_label', '')} of {rec.get('entity_label', '')} is {rec.get('object_label', '')}."
    return ar, en, fully

def featurise(scored):
    if not scored:
        return None
    e = np.array([s[0] for s in scored])
    c = np.array([s[1] for s in scored])
    n = np.array([s[2] for s in scored])

    es = np.sort(e)[::-1]
    cs = np.sort(c)[::-1]
    top = lambda a, k: float(a[k]) if len(a) > k else 0.0

    return [
        float(e.max()), float(e.mean()), top(es, 1), top(es, 2),        # support evidence
        float(c.max()), float(c.mean()), top(cs, 1), top(cs, 2),        # refute evidence
        float(n.mean()),                                                # how neutral overall
        float((e > 0.5).sum()), float((c > 0.5).sum()),                 # counts of strong passages
        float(len(scored)),                                             # page length
        float(e.max() - c.max())                                        # support vs refute margin
    ]

# 3. Score every page against both claims
rows = []
for i, rec in enumerate(data):
    gold = derived_gold(rec)
    if gold is None:
        continue

    ar_claim, en_claim, fully = claims(rec)
    path = os.path.join(HTML, rec.get('html', ''))
    sc_ar, sc_en = [], []

    try:
        if os.path.exists(path):
            html = open(path, 'rb').read()
            text = clean_to_text(html)
            passages = segment(text, detect_lang(text))
            for p in passages[:150]:
                a = entail_scores(p, ar_claim)
                e = entail_scores(p, en_claim)
                sc_ar.append((a.get('entailment', 0), a.get('contradiction', 0), a.get('neutral', 0)))
                sc_en.append((e.get('entailment', 0), e.get('contradiction', 0), e.get('neutral', 0)))
    except Exception:
        sc_ar, sc_en = [], []

    rows.append({
        'gold': gold,
        'fully': fully,
        'feats_ar': featurise(sc_ar),
        'feats_en': featurise(sc_en)
    })

    if (i + 1) % 50 == 0:
        print(f'  {i + 1}/{len(data)}')

output_path = f'{PD}/wtr_claim_features_bilingual.json'
json.dump(rows, open(output_path, 'w'))

print('\ncached ->', output_path)
print('references:', len(rows), '| fully-Arabic:', sum(r['fully'] for r in rows))
print('gold (fully-Arabic subset):', Counter(r['gold'] for r in rows if r['fully']))

In [ ]:
import json, numpy as np
from collections import Counter
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict

PD = PD
rows = json.load(open(f'{PD}/wtr_claim_features_bilingual.json'))
sub = [r for r in rows if r['fully'] and r['feats_ar'] and r['feats_en']]
print(f'controlled subset: {len(sub)} references |', Counter(r['gold'] for r in sub))

def run(key, title):
    X = np.array([r[key] for r in sub])
    y = np.array([r['gold'] for r in sub])
    classes = sorted(set(y))
    if len(sub) < 30 or len(classes) < 2:
        print(f'{title}: too few references to cross-validate')
        return
    k = min(5, min(Counter(y).values()))  # folds can't exceed smallest class
    if k < 2:
        print(f'{title}: a class has <2 examples — cannot stratify')
        return
    rf = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                min_samples_leaf=2, random_state=42)
    cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    pred = cross_val_predict(rf, X, y, cv=cv)
    prob = cross_val_predict(rf, X, y, cv=cv, method='predict_proba')
    si = classes.index('SUPPORTS') if 'SUPPORTS' in classes else 0
    evaluate([{'gold': g, 'pred': p, 'support_prob': float(pr[si])}
              for g, p, pr in zip(y, pred, prob)], title=title)

run('feats_en', 'CLAIM-LEVEL + RF — ENGLISH claim [fully-AR subset]')
run('feats_ar', 'CLAIM-LEVEL + RF — ARABIC claim  [fully-AR subset]')